# Topic modelling with BERTopic

**Day 2 afternoon — Introduction to Machine Learning for Text Analysis with Python**

<img src="../images/02_tokenization.png" alt="Tokenization brick" style="max-width: 130px;"> <img src="../images/03_vectorization.png" alt="Vectorization brick" style="max-width: 130px;"> <img src="../images/04_dimred_cluster.png" alt="Dimensionality reduction and clustering brick" style="max-width: 130px;"> <img src="../images/05_modelling.png" alt="Modelling brick" style="max-width: 130px;"> <img src="../images/06_evaluation.png" alt="Evaluation brick" style="max-width: 130px;">

This morning you built an unsupervised pipeline:  TF-IDF vectors -> PCA -> K-means -> top words per cluster. 

**BERTopic** follows largely the same procedural 'recipe', with some important differences: 

| Step | This morning | This afternoon |
|---|---|---|
| Represent | TF-IDF (i.e., word counts) | Transformer embeddings (i.e., meaning) |
| Reduce | PCA | UMAP |
| Cluster | K-means | HDBSCAN |
| Describe | Top words per cluster centre | c-TF-IDF per topic |


We stay with the full 20 Newsgroups collection. At the end we repeat this morning's 'benchmark': compare the discovered topics against the real newsgroups. This morning's result (to beat) was **ARI 0.147**.

We will use a pre-trained embedding model: `intfloat/multilingual-e5-small` ([model card](https://huggingface.co/intfloat/multilingual-e5-small)). See Section A3 for more on this.

The notebook has three parts. **Part A** is the main walk-through of the main pipeline. **Part B** are some more advanced options with the BERTopic pipeline to explore. **Part C** is a scaffold for applying BERTopic to your own data.

Reference: the [BERTopic documentation](https://maartengr.github.io/BERTopic/). It is excellent; bookmark it.

<img src="../images/bert_stack.svg" alt="BERTopic stack" style="max-width: 400px;">

The one part of this diagram we have not met is **c-TF-IDF** ("class-based TF-IDF"). After clustering, BERTopic concatenates all documents in a topic into one long pseudo-document. It then computes TF-IDF across these pseudo-documents, instead of across individual posts. A word scores high for a topic if it is frequent in that topic and rare in the others. These high-scoring words become the topic's description.

# Part A — main pipeline

## A1. Setup

In [ ]:
# the dataset (same as earlier)
from sklearn.datasets import fetch_20newsgroups

# BERTopic and other tools we'll need
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer   # we load the embedding model using this (to get document embeddings).
import torch                                            # this allows SentenceTransformer to use your CPU/GPU efficiently.
from umap import UMAP                                   # for dimensionality reduction (per this morning, but diff. technique)
from hdbscan import HDBSCAN                             # for clustering (per this morning, but diff. technique)
from sklearn.feature_extraction.text import CountVectorizer  # needed for BERTopic's c-TF-IDF method (topic descriptions).

# general tools
import numpy as np # for numerical operations
import pandas as pd # for dataframes

SEED = 42 #of course, we set a seed for reproducibility.

## A2. Load the data

<img src="../images/01_preprocessing.png" alt="Preprocessing brick" style="max-width: 130px;">

Same steps as this morning: all 20 newsgroups, headers/footers/quotes removed, near-empty posts dropped. We keep the newsgroup labels hidden so that we can benchmark the topic model afterwards. 


In [ ]:
newsgroups = fetch_20newsgroups(
    subset='all', #as with this morning, we'll use all the data (train + test), because we are training an unsupervised model.
    remove=('headers', 'footers', 'quotes'), #again, we drop these components so the model learns from the body and not from shortcuts. 
    random_state=SEED, #we pass our random seed so that the shuffled order of the data is reproducible. 
)

# drop near-empty posts, keeping each post paired with its newsgroup label
pairs = [(doc, label) for doc, label in zip(newsgroups.data, newsgroups.target) #
         if len(doc.strip()) > 50]
docs = [doc for doc, label in pairs]
labels = np.array([label for doc, label in pairs])
label_names = newsgroups.target_names

print(f"Number of posts: {len(docs)}")

## A3. From text to numbers, again — but with embeddings

<img src="../images/02_tokenization.png" alt="Tokenization brick" style="max-width: 130px;"> <img src="../images/03_vectorization.png" alt="Vectorization brick" style="max-width: 130px;">

This morning we represented each document (i.e., each post) as a vector of word counts. This meant that two documents were similar if they shared words. 

An **embedding model** works differently. It maps each document to a dense vector (a high dimension set of coordinates) that represents the document's meaning in 'semantic space'. It is able to do this quite well, because it is trained on enormous amounts of (historical) text. Documents with similar meanings are located close together in embedding space, even when they share no words. "The match went to overtime" and "a tight game decided late" may be located close together in embedding space because they bot likely regard sports; even though they share no words in common. 

We will use [`intfloat/multilingual-e5-small`](https://huggingface.co/intfloat/multilingual-e5-small). It is relatively small (118M parameters) and covers ~100 languages, so you can also use is on (your potentially) non-English data. Two practicalities:

- **Device.** Embedding is much faster on a GPU (`cuda` on Colab, `mps` on Apple Silicon) than on CPU. This is because a GPU is specialized for performing lots of simultaneous calculations (e.g., matrix operations). The cell below picks the best available device automatically.
- **Prefix.** Most modern embedding models (like this one) require that you add a prefix to the documents you encode. This is because they are 'fine-tuned' for specific commonly used downstream tasks. For our purposes, every text should start with `"query: "`. This requirement is in the model card (we'll discuss these cards and how to read them on Day 5). Forgetting the prefix will just mean it performs a bit worse at the task at hand.

> ⚠️ Embedding models read a limited window of text (here 512 tokens, roughly 350-400 words). Anything beyond the window is ignored. For posts and news articles this is usually fine; the most important information tends to come first. For long documents, split ("chunk") the text and embed the pieces.

In [ ]:
# pick the best available device (GPU > MPS > CPU)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# load the embedding model into SentenceTransformer (and pass it the device picked above)
embedding_model = SentenceTransformer("intfloat/multilingual-e5-small", device=device)

# register the prefix as the default prompt, so every encode() call applies it
embedding_model.prompts = {"e5": "query: "}
embedding_model.default_prompt_name = "e5"

In [ ]:
# we now encode (embed) the documents using the embedding model. 
# if you are on CPU, and this is taking > 10 mins, you may want to switch to Colab with a GPU runtime)
embeddings = embedding_model.encode(docs, show_progress_bar=True)

# finish by reporting the shape of the embeddings array (should be 18846 x 384, i.e., 18846 docs, each represented by a 384-dim vector)
print(f"Embeddings shape: {embeddings.shape}")

Each post is now a point in 384-dimensional space. Importantly, this is a dense space (every dimension has a continuous value), so a lot more information is encoded for each document than was the case this morning (where our bag-of-words produced a sparse document-term-matrix of mostly zeroes). 

## A4. Reduce and cluster

<img src="../images/04_dimred_cluster.png" alt="Dimensionality reduction and clustering brick" style="max-width: 130px;">

**UMAP** replaces PCA from this morning. It is a good standard choice for dimentional reduction for text data. UMAP uses a spatial approach (nearest neighbours), and builds a low-dimensional layout that preserves the neighbourhood structure of the higher-dimension data. This suits the curved, uneven shape of embedding spaces. It also means we can reduce all the way down to 5 dimensions and still retain plenty of information for the clustering to operate with. 

**HDBSCAN** replaces K-means's from this morning. It fixes the two problems we encountered:

1. **It chooses the number of clusters itself.** You set a *minimum cluster size*; it finds however many dense groups exist at that scale.
2. **It can leave a document unassigned.** K-means forces every document into a cluster. HDBSCAN leaves documents outside any dense region unassigned. These outliers get the label `-1`. With messy real-world text, there can be many.

In [ ]:
# UMAP: spatial-based dimensionality reduction; reduce 384 dimensions to 5
umap_model = UMAP(n_neighbors=15, n_components=5,
                  min_dist=0.0, metric='cosine', random_state=SEED)

# HDBSCAN: density-based clustering; min_cluster_size indirectly controls
# how many topics you get (larger value -> fewer, bigger topics)
hdbscan_model = HDBSCAN(min_cluster_size=60, metric='euclidean',
                        cluster_selection_method='eom', prediction_data=True)

# vectorizer for the topic DESCRIPTIONS (c-TF-IDF works on word counts, so
# stopword removal is useful HERE)
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

## A5. Fit the model

<img src="../images/05_modelling.png" alt="Modelling brick" style="max-width: 130px;">

BERTopic allows us to pass the various parts of the 'stack' we just created. 
We take care to pass in our precomputed embeddings, so it does not embed everything again.

In [ ]:
topic_model = BERTopic(
    embedding_model=embedding_model,    # step 1: embed (skipped, we pass ours in below)
    umap_model=umap_model,              # step 2: reduce (using the UMAP spec above)
    hdbscan_model=hdbscan_model,        # step 3: cluster (using the HDBSCAN spec above)
    vectorizer_model=vectorizer_model,  # step 4: describe topics (using the c-TF-IDF spec above)
    verbose=True,                       # this will print out progress messages as the model runs
)

topics, probs = topic_model.fit_transform(docs, embeddings) #this is where we pass our pre-computed embeddings.

## A6. Explore the topics

<img src="../images/06_evaluation.png" alt="Evaluation brick" style="max-width: 130px;">

`get_topic_info` gives one row per topic: id, size, and describing words. Topic `-1` collects outliers (it is not a real topic).

In [ ]:
topic_model.get_topic_info()

In [ ]:
# the full c-TF-IDF word list (with scores) for one topic:
topic_model.get_topic(0)

In [ ]:
# which topic did a given document get, and does it make sense? always spot-check:
i = 0
print(f"Document {i} was assigned topic {topics[i]}:")
print(topic_model.get_topic(topics[i])[:5], "\n")
print(docs[i][:500])

BERTopic comes packaged with various interactive visualisations. 

The bar chart (below) shows the top words of the largest topics. The hierarchy (also below) shows which topics are similar (based on their c-TF-IDF vectors) and would merge first.

In [ ]:
topic_model.visualize_barchart(top_n_topics=12)

In [ ]:
topic_model.visualize_hierarchy()

We can also visualize the seoamtic location of every post in 2D, coloured by topic. For this plot we run a second UMAP, down to 2 dimensions.

In [ ]:
umap_2d = UMAP(n_neighbors=15, n_components=2, min_dist=0.0,
               metric='cosine', random_state=SEED)
reduced_embeddings = umap_2d.fit_transform(embeddings)

topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings)

We can also create a 3D visualization. Notice that this already 

## A7. Did we manage to recover the topics better than we did this morning? 

This morning K-means reached an ARI of 0.147 against the true newsgroups. Did we do better? 

- HDBSCAN chose its own number of topics, which need not be 20. This is fine: ARI compares groupings without requiring the same number of groups.
- The outliers (topic `-1`) have no counterpart in the morning's setup. We therefore compute the score twice. Over all documents, with the outliers counted as one big group, which lowers the score. And over assigned documents only, which reflects clustering quality better, but on an easier subset.

In [ ]:
from sklearn.metrics import adjusted_rand_score

topics_arr = np.array(topics)
assigned = topics_arr != -1

print(f"Share of documents assigned to a topic: {assigned.mean():.0%}")
print(f"ARI, all documents:      {adjusted_rand_score(labels, topics_arr):.3f}")
print(f"ARI, assigned docs only: {adjusted_rand_score(labels[assigned], topics_arr[assigned]):.3f}")
print(f"(this morning, TF-IDF + PCA + K-means: 0.147)")

In [ ]:
# and the heatmap: true newsgroups vs discovered topics (outliers excluded)
import seaborn as sns
import matplotlib.pyplot as plt

crosstab = pd.crosstab(
    pd.Series([label_names[l] for l in labels[assigned]], name='true newsgroup'),
    pd.Series(topics_arr[assigned], name='topic'),
)
plt.figure(figsize=(12, 6))
sns.heatmap(crosstab, cmap='Blues', cbar=False)
plt.title('True newsgroups vs BERTopic topics (assigned documents)')
plt.tight_layout()
plt.show()

The embedding-based pipeline beats the morning's score by a wide margin. Embeddings produce a much richer semantic description of texts. This is the main reason transformer embeddings have displaced bag-of-words methods for most ML workflows (especially inductive / exploratory work).

As with this morning, some newsgroups split across several topics. That is often a finding, as for example one discussion group can host several distinguishable sub-topics that overlap between different newsgroups in this case. There are also a large number of outliers: about a quarter of the corpus was set aside rather than forced into a topic. Whether that is acceptable depends on your research question. Part B2 below explores this further. 

---

# Part B — extensions

Note: Each section below can be run indepedently (no need to run in order, so you can pick what is most interesting to you). 
All parts reuse `docs`, `embeddings`, and the fitted `topic_model` from Part A.

## B1. Better topic descriptions

The default topic words come from c-TF-IDF alone, and often include filler. Two refiners help. BERTopic lets you chain them:

- **KeyBERTInspired** re-ranks candidate words by comparing each word's embedding with the topic's embedding (essentially the 'middle' of the topic in semantic space). Words whose meaning is closer to that of the topic rise to the top.
- **MaximalMarginalRelevance (MMR)** then selects a *diverse* subset, so the list is not five variants of the same word.

Topic descriptions can also be written by a generative LLM (see BERTopic docs for this).

In [ ]:
import copy
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

representation_model = [
    KeyBERTInspired(top_n_words=30), 
    MaximalMarginalRelevance(diversity=0.3),
]

# try the new representation on a COPY of the fitted model, so topic_model
# itself is left untouched -- re-run this cell with different settings as
# often as you like. update_topics only rewrites the topic DESCRIPTIONS; the
# clusters themselves stay exactly as they are.
trial_model = copy.deepcopy(topic_model)
trial_model.update_topics(docs, vectorizer_model=vectorizer_model,
                          representation_model=representation_model)

# old and new descriptions side by side (Representative_Docs dropped to make room)
old_info = topic_model.get_topic_info()[["Topic", "Count", "Name", "Representation"]]
new_info = trial_model.get_topic_info()[["Topic", "Representation"]]
comparison = old_info.merge(new_info, on="Topic", suffixes=("_old", "_new"))
comparison.head(12)


## B2. Rescuing the outliers

Some research questions need every document categorised — for example, topic proportions over the whole corpus. In that case you can reassign the outliers to their nearest topic after fitting. The `embeddings` strategy gives each outlier the topic whose embedding is closest to its own.

This has a cost. These documents were outliers *because* they sat in no dense region. Forcing them into topics potentially adds noise to those topics. In this case, because we have the labels (the newsgroups), we can re-measure the ARI to see what this 'costs':

In [ ]:
new_topics = topic_model.reduce_outliers(docs, topics, strategy="embeddings",
                                         embeddings=embeddings)

new_topics_arr = np.array(new_topics)
print(f"Outliers before: {(topics_arr == -1).sum()}")
print(f"Outliers after:  {(new_topics_arr == -1).sum()}\n")

print(f"ARI over all documents, before rescue: {adjusted_rand_score(labels, topics_arr):.3f}")
print(f"ARI over all documents, after rescue:  {adjusted_rand_score(labels, new_topics_arr):.3f}")

Here the reassignment helped! The score over all documents rose. Many outliers were in fact located close to a sensible topic. To keep the new assignments, update the model so the topic descriptions reflect their new members: `topic_model.update_topics(docs, topics=new_topics)`.

## B3. Controlling the number of topics

Three ways to control the number of topics:

1. **`min_cluster_size`** (before fitting). A bigger minimum gives fewer, larger topics. Refit with 300 instead of 60 and see what happens.
2. **`reduce_topics`** (after fitting). Merges the least distinct topics until a target count remains, following the hierarchy from A6.
3. **`delete_topics`** (after fitting). Removes specific topics you judge to be junk. Their documents become outliers.

In [ ]:
# merge down to 20 topics (matching, for once, what we know about this corpus)
topic_model.reduce_topics(docs, nr_topics=21)   # 21 = 20 topics + the outlier pile
topics_reduced = np.array(topic_model.topics_)

print(f"ARI after merging to 20 topics (assigned docs only): "
      f"{adjusted_rand_score(labels[topics_reduced != -1], topics_reduced[topics_reduced != -1]):.3f}")
topic_model.get_topic_info().head(10)

Merging down to exactly 20 topics lowered the ARI slightly, meaning that the topic model's boundaries differ from those of the newsgroups. 

## B4. Steering the model

Everything so far was purely inductive. Often you know something in advance. BERTopic has three ways to feed that knowledge in in influence the resulting topic model: 

- **Seed words.** Nudge the topic *descriptions*  toward domain vocabulary. This does NOT affect the clustering (i.e, the topics), only their descriptions. 
- **Zero-shot topic modelling.** You name the topics you expect. Documents that match a name closely enough (in semantic space) are assigned to it. Only the remaining documents go through the usual inductive clustering. 
- **Semi-supervised modelling.** You pass known labels for some or all documents to `fit` (the `y` argument). The reduction step then pulls documents with the same label together before clustering.

Zero-shot is the most interesting for us. We can hand the model the actual newsgroup themes as candidate topics, and inspect the deductive/inductive split.

In [ ]:
# describe the expected topics in plain language (descriptive names work better
# than the cryptic newsgroup names themselves)
zeroshot_topics = [
    "computer graphics and image software",
    "microsoft windows operating system",
    "pc hardware, motherboards and drives",
    "apple macintosh hardware",
    "cars and driving",
    "motorcycles and riding",
    "baseball",
    "ice hockey",
    "cryptography and encryption",
    "electronics and circuits",
    "medicine and health",
    "space flight and astronomy",
    "christianity and faith",
    "gun politics and firearms",
    "middle east politics",
    "atheism and religion debates",
    "items for sale",
]

# component objects store their fitted state, so a new model needs its own
# fresh instances -- reusing the ones inside topic_model would corrupt it
zeroshot_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                    metric='cosine', random_state=SEED),
    hdbscan_model=HDBSCAN(min_cluster_size=60, metric='euclidean',
                          cluster_selection_method='eom', prediction_data=True),
    vectorizer_model=CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2)),
    zeroshot_topic_list=zeroshot_topics,
    zeroshot_min_similarity=0.80,   # tune this: e5 similarities run high
    verbose=True,
)
zs_topics, _ = zeroshot_model.fit_transform(docs, embeddings)

zeroshot_model.get_topic_info()

In the topic table, the topics carrying your plain-language names were filled deductively. The numbered topics below them were found inductively, among the documents that did not meet the similarity criteria (> 0.80 cosine similarity). Both the similarity threshold and the wording of your topic names matter. 

## B5. Working with a fitted model

Here are three useful additional tools for exploring your topic model. First, semantic topic search. `find_topics` embeds your query and returns the most similar topics. Useful once a model has dozens of topics. Let's try for 'religion and belief':

In [ ]:
topic_ids, similarities = topic_model.find_topics("religion and belief", top_n=3)
for tid, sim in zip(topic_ids, similarities):
    print(f"topic {tid} (similarity {sim:.2f}): {[w for w, _ in topic_model.get_topic(tid)[:6]]}")

Second, topic *distributions*. Hard assignment (one topic per document) is of course a reduction; a post can be about hockey *and* medicine. `approximate_distribution` slides a window over each document (default: 4 tokens wide, moving 1 token at a time) and scores every topic's share of each window. Summed over the windows, this gives a soft distribution per document.


In [ ]:
N = 2000   # the windowed scoring is quick, but we don't need the whole corpus
topic_distr, topic_token_distr = topic_model.approximate_distribution(
    docs[:N], calculate_tokens=True)

# the distribution over topics for one document:
i = 1   # try a few different documents
print(f"document {i}, assigned topic {topics[i]}")
topic_model.visualize_distribution(topic_distr[i])


Most posts are dominated by one topic, so the bar chart mostly confirms the hard assignment. The mixing shows at the level of passages. With `calculate_tokens=True` we also got a score per token, and BERTopic can colour the document by topic:


In [ ]:
# BERTopic 0.17 still calls Styler.applymap, which pandas 3 renamed to .map
from pandas.io.formats.style import Styler
Styler.applymap = Styler.map

topic_model.visualize_approximate_distribution(docs[i], topic_token_distr[i])


**Across the corpus.** The practical reason for soft distributions is aggregation, for instance topic proportions per source, period, or group. Here we average each document's distribution within its true newsgroup. The diagonal shows that the topics track the newsgroups; the off-diagonal mass shows which groups share content.


In [ ]:
topic_ids = topic_model.get_topic_info().query("Topic != -1")["Topic"].tolist()
distr_df = pd.DataFrame(topic_distr, columns=topic_ids)
distr_df["newsgroup"] = [label_names[l] for l in labels[:N]]
proportions = distr_df.groupby("newsgroup").mean()

plt.figure(figsize=(12, 7))
sns.heatmap(proportions, cmap="Blues", cbar_kws={"label": "mean topic share"})
plt.title("Average topic distribution per true newsgroup")
plt.xlabel("topic")
plt.tight_layout()
plt.show()


Third, saving. A fitted model can be stored and reloaded without refitting. You can also share the saved model, or push it to the Hugging Face Hub.

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "my_topic_model")
    topic_model.save(path, serialization="safetensors", save_ctfidf=True,
                     save_embedding_model="intfloat/multilingual-e5-small")
    reloaded = BERTopic.load(path)
    print(f"Reloaded model has {len(reloaded.get_topic_info()) - 1} topics")

---

# Part C — your own data

Below is a scaffold for running BERTopic on your own data. You will need to input it as a list of strings called `my_docs`, one string per document. 

Check three things before you run it:

- **Language.** Not a problem: e5-small covers ~100 languages.
- **Document length.** Remember the 512-token window. If your documents are long (transcripts, full articles, parliamentary speeches), you may want to chunk them into paragraphs first and treat each chunk as a document, or mean pool (take the average of) the chunks to represent the document. 
- **`min_cluster_size`.** There is no universal rule. 60 suited our ~18,000 posts. For a smaller corpus, 1-3% of your document count is a reasonable start, but do experiment. 

In [ ]:
# load your data -- e.g. from a CSV:
# my_df = pd.read_csv("my_data.csv")
# my_docs = my_df["text"].dropna().tolist()

my_docs = []  # <-- your documents here

if my_docs:
    my_embeddings = embedding_model.encode(my_docs, show_progress_bar=True)

    my_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                        metric='cosine', random_state=SEED),
        hdbscan_model=HDBSCAN(min_cluster_size=max(10, len(my_docs) // 40),
                              metric='euclidean', cluster_selection_method='eom',
                              prediction_data=True),
        vectorizer_model=CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2)),
        verbose=True,
    )
    my_topics, _ = my_model.fit_transform(my_docs, my_embeddings)
    display(my_model.get_topic_info())

If the topics look poor, work through the levers in this order: read actual documents per topic (is the problem real or cosmetic?); adjust `min_cluster_size`; improve the descriptions as in B1; only then touch the UMAP parameters. If your documents are English-only and you want faster embedding, swap in `all-MiniLM-L6-v2`. It is lighter and needs no prefix, so also drop the prompt lines.